# Lab 8: Implementation and Performance Evaluation of Categorical Naive Bayes Classifier

This notebook implements a classification pipeline using the standard Play Tennis dataset. It covers data preprocessing, dataset partitioning, Naive Bayes model training and evaluation, single-sample inference, and a comparison with other classification models.

## 1. Data Preprocessing

In [12]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# Load the dataset
url = 'https://docs.google.com/spreadsheets/d/1mnoUgK8YxInzoDQ7P7iBM1HeZ8tcnUSvMU_H1OWndFA/gviz/tq?tqx=out:csv&gid=0'
df = pd.read_csv(url)
df = df.drop(columns=['No'])

print("Original Dataset Head:")
display(df.head())
print("\nDataset Info:")
df.info()

Original Dataset Head:


,Outlook,Temperature,Humidity,Wind,Play Tennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Outlook      50 non-null     object
 1   Temperature  50 non-null     object
 2   Humidity     50 non-null     object
 3   Wind         50 non-null     object
 4   Play Tennis  50 non-null     object
dtypes: object(5)
memory usage: 2.1+ KB


In [13]:
# Separate input features (X) from the target variable (y)
X = df.drop(columns=['Play Tennis'])
y = df['Play Tennis']

print("Features (X) Head:")
display(X.head())
print("\nTarget (y) Head:")
display(y.head())

Features (X) Head:


,Outlook,Temperature,Humidity,Wind
0,Sunny,Hot,High,Weak
1,Sunny,Hot,High,Strong
2,Overcast,Hot,High,Weak
3,Rain,Mild,High,Weak
4,Rain,Cool,Normal,Weak



Target (y) Head:


,Play Tennis
0,No
1,No
2,Yes
3,Yes
4,Yes


In [14]:
# Convert all categorical feature values and target labels into numerical representations
# Using OrdinalEncoder for features
encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(X)

# Convert to DataFrame to maintain column names
X_encoded_df = pd.DataFrame(X_encoded, columns=X.columns)

# Encode target variable y
y_encoded = y.astype('category').cat.codes

# Map original target labels to encoded values for clarity in results
target_mapping = dict(enumerate(y.astype('category').cat.categories))
print(f"Target Variable Mapping: {target_mapping}")

print("\nEncoded Features (X_encoded_df) Head:")
display(X_encoded_df.head())
print("\nEncoded Target (y_encoded) Head:")
display(y_encoded.head())

Target Variable Mapping: {0: 'No', 1: 'Yes'}

Encoded Features (X_encoded_df) Head:


,Outlook,Temperature,Humidity,Wind
0,2.0,1.0,0.0,1.0
1,2.0,1.0,0.0,0.0
2,0.0,1.0,0.0,1.0
3,1.0,2.0,0.0,1.0
4,1.0,0.0,1.0,1.0



Encoded Target (y_encoded) Head:


,0
0,0
1,0
2,1
3,1
4,1


## 2. Dataset Partitioning

In [15]:
from sklearn.model_selection import train_test_split

# Divide the dataset into training and testing subsets (80:20 ratio)
X_train, X_test, y_train, y_test = train_test_split(X_encoded_df, y_encoded, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")

Training set size: 40 samples
Testing set size: 10 samples


## 3. Naive Bayes Model Training & Evaluation

In [16]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Train a Categorical Naive Bayes (CategoricalNB) model on the training data
cnb_model = CategoricalNB()
cnb_model.fit(X_train, y_train)

# Predict the class labels for the test dataset
y_pred_cnb = cnb_model.predict(X_test)

# Calculate and display the overall Model Accuracy
print("Categorical Naive Bayes Classifier Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_cnb):.4f}")

# Display the Confusion Matrix
print("\nConfusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_cnb), index=target_mapping.values(), columns=target_mapping.values()))

# Display the Classification Report (Precision, Recall, F1-Score)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_cnb, target_names=target_mapping.values()))

Categorical Naive Bayes Classifier Evaluation:
Accuracy: 0.8000

Confusion Matrix:


,No,Yes
No,1,1
Yes,1,7



Classification Report:
              precision    recall  f1-score   support

          No       0.50      0.50      0.50         2
         Yes       0.88      0.88      0.88         8

    accuracy                           0.80        10
   macro avg       0.69      0.69      0.69        10
weighted avg       0.80      0.80      0.80        10



## 4. Single-Sample Inference (Categorical Naive Bayes)

In [17]:
# Predict whether a person will play tennis under specific weather conditions:
# Outlook: Sunny, Temperature: Cool, Humidity: High, Wind: Strong

# Based on the encoding from X_encoded_df:
# Outlook: Sunny -> 2
# Temperature: Cool -> 0
# Humidity: High -> 0
# Wind: Strong -> 0
single_sample_data = pd.DataFrame([[2.0, 0.0, 0.0, 0.0]], columns=X_encoded_df.columns)

predicted_label_cnb_single = cnb_model.predict(single_sample_data)
predicted_proba_cnb_single = cnb_model.predict_proba(single_sample_data)

print(f"Single-Sample Query: Outlook: Sunny, Temperature: Cool, Humidity: High, Wind: Strong")
print(f"Predicted Class Label: {target_mapping[predicted_label_cnb_single[0]]}")
print(f"Corresponding Class Probabilities: {predicted_proba_cnb_single[0]}")

Single-Sample Query: Outlook: Sunny, Temperature: Cool, Humidity: High, Wind: Strong
Predicted Class Label: No
Corresponding Class Probabilities: [0.92560203 0.07439797]


### Decision Tree Classifier

In [20]:
from sklearn.tree import DecisionTreeClassifier

# Train a Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

# Predict on the test dataset
y_pred_dt = dt_model.predict(X_test)

# Evaluate the Decision Tree model
print("Decision Tree Classifier Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print("\nConfusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_dt), index=target_mapping.values(), columns=target_mapping.values()))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt, target_names=target_mapping.values()))

# Single-Sample Inference
# Outlook: Sunny, Temperature: Cool, Humidity: High, Wind: Strong
# Based on the mapping used for X_encoded_df:
# Outlook: Sunny -> 2
# Temperature: Cool -> 0
# Humidity: High -> 0
# Wind: Strong -> 0
single_sample_dt = pd.DataFrame([[2.0, 0.0, 0.0, 0.0]], columns=X_encoded_df.columns)
predicted_label_dt = dt_model.predict(single_sample_dt)
predicted_proba_dt = dt_model.predict_proba(single_sample_dt)

print(f"\nSingle-Sample Prediction (Decision Tree): {target_mapping[predicted_label_dt[0]]}")
print(f"Class Probabilities: {predicted_proba_dt[0]}")

Decision Tree Classifier Evaluation:
Accuracy: 0.8000

Confusion Matrix:


,No,Yes
No,1,1
Yes,1,7



Classification Report:
              precision    recall  f1-score   support

          No       0.50      0.50      0.50         2
         Yes       0.88      0.88      0.88         8

    accuracy                           0.80        10
   macro avg       0.69      0.69      0.69        10
weighted avg       0.80      0.80      0.80        10


Single-Sample Prediction (Decision Tree): No
Class Probabilities: [1. 0.]


### Logistic Regression Classifier

In [21]:
from sklearn.linear_model import LogisticRegression

# Train a Logistic Regression Classifier
lr_model = LogisticRegression(random_state=42, solver='liblinear')
lr_model.fit(X_train, y_train)

# Predict on the test dataset
y_pred_lr = lr_model.predict(X_test)

# Evaluate the Logistic Regression model
print("Logistic Regression Classifier Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print("\nConfusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_lr), index=target_mapping.values(), columns=target_mapping.values()))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=target_mapping.values()))

# Single-Sample Inference
single_sample_lr = pd.DataFrame([[2.0, 0.0, 0.0, 0.0]], columns=X_encoded_df.columns)
predicted_label_lr = lr_model.predict(single_sample_lr)
predicted_proba_lr = lr_model.predict_proba(single_sample_lr)

print(f"\nSingle-Sample Prediction (Logistic Regression): {target_mapping[predicted_label_lr[0]]}")
print(f"Class Probabilities: {predicted_proba_lr[0]}")

Logistic Regression Classifier Evaluation:
Accuracy: 0.4000

Confusion Matrix:


,No,Yes
No,0,2
Yes,4,4



Classification Report:
              precision    recall  f1-score   support

          No       0.00      0.00      0.00         2
         Yes       0.67      0.50      0.57         8

    accuracy                           0.40        10
   macro avg       0.33      0.25      0.29        10
weighted avg       0.53      0.40      0.46        10


Single-Sample Prediction (Logistic Regression): No
Class Probabilities: [0.9466476 0.0533524]


### Support Vector Machine (SVM) Classifier

In [22]:
from sklearn.svm import SVC

# Train an SVM Classifier (with probability estimates enabled)
svm_model = SVC(random_state=42, probability=True)
svm_model.fit(X_train, y_train)

# Predict on the test dataset
y_pred_svm = svm_model.predict(X_test)

# Evaluate the SVM model
print("SVM Classifier Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")
print("\nConfusion Matrix:")
display(pd.DataFrame(confusion_matrix(y_test, y_pred_svm), index=target_mapping.values(), columns=target_mapping.values()))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm, target_names=target_mapping.values()))

# Single-Sample Inference
single_sample_svm = pd.DataFrame([[2.0, 0.0, 0.0, 0.0]], columns=X_encoded_df.columns)
predicted_label_svm = svm_model.predict(single_sample_svm)
predicted_proba_svm = svm_model.predict_proba(single_sample_svm)

print(f"\nSingle-Sample Prediction (SVM): {target_mapping[predicted_label_svm[0]]}")
print(f"Class Probabilities: {predicted_proba_svm[0]}")

SVM Classifier Evaluation:
Accuracy: 0.7000

Confusion Matrix:


,No,Yes
No,0,2
Yes,1,7



Classification Report:
              precision    recall  f1-score   support

          No       0.00      0.00      0.00         2
         Yes       0.78      0.88      0.82         8

    accuracy                           0.70        10
   macro avg       0.39      0.44      0.41        10
weighted avg       0.62      0.70      0.66        10


Single-Sample Prediction (SVM): No
Class Probabilities: [0.97514137 0.02485863]


### Overall Model Comparison Summary

Let's compile the results from all four classifiers (Categorical Naive Bayes, Decision Tree, Logistic Regression, and SVM) for a comprehensive comparison.

In [23]:
import pandas as pd

# Collect accuracies
accuracies = {
    "Categorical Naive Bayes": accuracy_score(y_test, y_pred_cnb),
    "Decision Tree": accuracy_score(y_test, y_pred_dt),
    "Logistic Regression": accuracy_score(y_test, y_pred_lr),
    "SVM": accuracy_score(y_test, y_pred_svm)
}

# Collect single-sample predictions and probabilities
single_sample_results = {
    "Categorical Naive Bayes": {
        "Prediction": target_mapping[predicted_label_cnb_single[0]],
        "Probabilities": predicted_proba_cnb_single[0]
    },
    "Decision Tree": {
        "Prediction": target_mapping[predicted_label_dt[0]],
        "Probabilities": predicted_proba_dt[0]
    },
    "Logistic Regression": {
        "Prediction": target_mapping[predicted_label_lr[0]],
        "Probabilities": predicted_proba_lr[0]
    },
    "SVM": {
        "Prediction": target_mapping[predicted_label_svm[0]],
        "Probabilities": predicted_proba_svm[0]
    }
}

print("\n--- Model Accuracy Comparison ---")
for model, acc in accuracies.items():
    print(f"{model}: {acc:.4f}")

print("\n--- Single-Sample Prediction Comparison (Outlook: Sunny, Temperature: Cool, Humidity: High, Wind: Strong) ---")
for model, result in single_sample_results.items():
    print(f"{model}: Predicted Label = {result['Prediction']}, Probabilities = {result['Probabilities']}")

# You can also create a DataFrame for a cleaner display
comparison_df = pd.DataFrame({
    'Model': list(accuracies.keys()),
    'Accuracy': list(accuracies.values()),
    'Single Sample Prediction': [res['Prediction'] for res in single_sample_results.values()],
    'Single Sample Probabilities': [str(res['Probabilities']) for res in single_sample_results.values()]
})

print("\n--- Detailed Comparison Table ---")
display(comparison_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True))


--- Model Accuracy Comparison ---
Categorical Naive Bayes: 0.8000
Decision Tree: 0.8000
Logistic Regression: 0.4000
SVM: 0.7000

--- Single-Sample Prediction Comparison (Outlook: Sunny, Temperature: Cool, Humidity: High, Wind: Strong) ---
Categorical Naive Bayes: Predicted Label = No, Probabilities = [0.92560203 0.07439797]
Decision Tree: Predicted Label = No, Probabilities = [1. 0.]
Logistic Regression: Predicted Label = No, Probabilities = [0.9466476 0.0533524]
SVM: Predicted Label = No, Probabilities = [0.97514137 0.02485863]

--- Detailed Comparison Table ---


,Model,Accuracy,Single Sample Prediction,Single Sample Probabilities
0,Categorical Naive Bayes,0.8,No,[0.92560203 0.07439797]
1,Decision Tree,0.8,No,[1. 0.]
2,SVM,0.7,No,[0.97514137 0.02485863]
3,Logistic Regression,0.4,No,[0.9466476 0.0533524]


Analysis Report:

The four classifiers produced different predictions and probability scores because each model uses a different learning approach and decision-making process. Categorical Naive Bayes assumes that all features are independent, whereas the Decision Tree classifies the instance using learned decision rules. Logistic Regression estimates probabilities using a linear decision boundary, while SVM finds the optimal separating hyperplane and computes probabilities differently. Therefore, even when the predicted class is the same, the confidence (probability scores) and accuracy of each model may vary due to their different underlying algorithms and assumptions.